In [ ]:
# !pip install torch torchvision matplotlib -q
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

**Load and Preprocess CIFAR-10 Dataset**


*   CIFAR-10 contains 32x32 RGB images belonging to 10 different classes.
*   DataLoader is used for efficient batch loading and shuffling during training.
*   Normalization is applied by standardizing pixel values.


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4), # Random crop means that we will randomly crop the image to 32x32 pixels, with a padding of 4 pixels on each side.
            # This means that the original image will be padded with 4 pixels on each side, and then a random 32x32 crop will be taken from the padded image. 
            # This is a common data augmentation technique used to increase the diversity of the training data and help prevent overfitting.
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),#Mean   These numbers are the average color values of the entire CIFAR-10 dataset.
                         (0.2023, 0.1994, 0.2010))#std   If you don't know the mean and standard deviation of the dataset, you can just use (0.5, 0.5, 0.5) for both mean and standard deviation, which will normalize the pixel values to the range [-1, 1].
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2) 
# batch_size = 128 → pick 128 images at a time
# batch_size = 32  → pick 32 images at a time
# shuffle=True → shuffle the data at every epoch, which helps to prevent overfitting and improve the generalization of the model. ( let's say all the images of planes are the first 1000 images.)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

1. why the batch_size is different in test and train? 
2. why the shuffle is off in test?

**Define the Residual Block**

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

**Define ResNet18 Architecture**


*   Four layers of residual blocks with increasing channels
*   Each layer is created with make layer function to handle multiple blocks
*   Global Average Pooling converts feature maps to vector
*   Fully Connected Layer outputs predictions for 10 classes

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

model = ResNet18().to(device)
print(model)

**Define Loss Function and Optimizer**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1) # Instead of manually adjusting the learning rate,
# we can use a learning rate scheduler to automatically adjust the learning rate during training. In this case, we are using StepLR, which decreases the learning rate by a factor of gamma every step_size epochs.
# So in this case, the learning rate will be decreased by a factor of 0.1 every 30 epochs. This helps to improve the convergence of the model and can lead to better performance.

## Momentum : 
Momentum helps the optimizer keep moving in the same good direction instead of changing direction too much.

Imagine pushing a ball down a hill 🟢

Without momentum → the ball stops and changes direction easily.

With momentum → the ball keeps rolling forward smoothly.

So in training:

momentum = 0.9 means
→ the optimizer remembers 90% of the previous movement direction.

Why it helps:

Faster training

Less zig-zagging

More stable learning

## weight_decay

Weight decay prevents the model from making weights too large.

Think of it like a small penalty for big weights.

Why?

Very large weights → model can memorize the training data

This leads to overfitting

So weight_decay:

keeps weights smaller

helps the model generalize better

**Train the model**

In [ ]:
# Number of times the model will see the entire training dataset
num_epochs = 10

# Lists to store metrics during training
train_losses = []      # stores training loss for each epoch
train_acc_list = []    # stores training accuracy for each epoch
test_acc_list = []     # stores test accuracy for each epoch


# Loop over the dataset multiple times (epochs)
for epoch in range(num_epochs):

    # Put the model in training mode (important for layers like Dropout and BatchNorm)
    model.train()

    # Variable to accumulate the total loss over the epoch
    running_loss = 0.0

    # Counters used to compute training accuracy
    correct = 0   # number of correct predictions
    total = 0     # total number of samples


    # Loop over batches of training data
    for inputs, labels in trainloader:

        # Move data to GPU or CPU depending on the device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Reset gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass: input images go through the model to produce predictions
        outputs = model(inputs)

        # Compute the loss between predictions and true labels
        loss = criterion(outputs, labels)

        # Backpropagation: compute gradients of the loss
        loss.backward()

        # Update model weights using the optimizer (SGD)
        optimizer.step()


        # Multiply loss by batch size and accumulate it
        running_loss += loss.item() * inputs.size(0)

        # Get the predicted class (index of highest probability)
        _, predicted = outputs.max(1)

        # Update total number of samples processed
        total += labels.size(0)

        # Count how many predictions are correct
        correct += predicted.eq(labels).sum().item()


    # Compute the average loss over the entire training dataset
    train_loss = running_loss / len(trainloader.dataset)

    # Compute training accuracy percentage
    train_acc = 100. * correct / total

    # Store the metrics for later analysis or plotting
    train_losses.append(train_loss)
    train_acc_list.append(train_acc)


    # Switch model to evaluation mode (important for BatchNorm/Dropout)
    model.eval()

    # Reset counters for test accuracy
    correct = 0
    total = 0


    # Disable gradient computation during testing (saves memory and computation)
    with torch.no_grad():

        # Loop over the test dataset
        for inputs, labels in testloader:

            # Move data to device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass (prediction)
            outputs = model(inputs)

            # Get predicted class
            _, predicted = outputs.max(1)

            # Update total samples
            total += labels.size(0)

            # Count correct predictions
            correct += predicted.eq(labels).sum().item()


    # Compute test accuracy
    test_acc = 100. * correct / total

    # Store test accuracy
    test_acc_list.append(test_acc)


    # Update the learning rate according to the scheduler
    scheduler.step()


    # Print training statistics for this epoch
    print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%')

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_acc_list, label='Train Accuracy')
plt.plot(test_acc_list, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy')
plt.legend()

plt.show()

Now let's make some predictions : 

In [ ]:
# Get one batch of images from the test loader
dataiter = iter(testloader)
images, labels = next(dataiter)
images = images.to(device)

# Disable gradients (not needed for inference)
with torch.no_grad():
    outputs = model(images)

# Get predicted class
_, predicted = torch.max(outputs, 1)

def imshow(img):
    img = img / 2 + 0.5     # unnormalize (approximate)
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis("off")

In [ ]:
# Move images back to CPU for visualization
images = images.cpu()

# Show first 6 images
plt.figure(figsize=(10,4))

for i in range(6):
    plt.subplot(1,6,i+1)
    imshow(images[i])
    plt.title(f"P:{classes[predicted[i]]}\nT:{classes[labels[i]]}")

can you interpret the new results?